# Clustering Wilayah untuk Deteksi Disparitas Ketepatan Sasaran Penyaluran BPNT dan PKH

**Nama:** Abraham Rusty Djajani  
**NIM:** 535240007  
**Email:** abraham.535240007@stu.untar.ac.id

**Data:** BPS (P0 2025, BPNT 2024 Sumsel, BPNT+PKH 2023 Jateng, BPNT+PKH 2021 Jakarta)

**Tujuan:**
1. Mengelompokkan wilayah berdasarkan indikator ketepatan sasaran bansos
2. Mengidentifikasi cluster under-coverage, over-coverage, dan tepat sasaran
3. Membandingkan K-Means, HAC (Ward), dan DBSCAN

---
## 1. Import Pustaka & Konfigurasi

In [1]:
import os, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

OUTPUT_DIR = "/Users/budydjajani/Documents/outputs/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

plt.rcParams.update({"figure.dpi": 150, "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold"})
CLUSTER_COLORS = ["#2563EB", "#DC2626", "#059669", "#D97706", "#7C3AED", "#6B7280"]
print(f"Output: {OUTPUT_DIR}")

DATA_DIR = "/Users/budydjajani/Documents"  # folder data Excel BPS

Output: /Users/budydjajani/Documents/outputs/outputs


---
## 2. Baca Data Excel BPS

In [2]:
# 2a. Persentase Penduduk Miskin (P0) 2025
f_p0 = os.path.join(DATA_DIR, "Persentase Penduduk Miskin (P0) Menurut Kabupaten_Kota, 2025.xlsx")
df_p0_raw = pd.read_excel(f_p0)
df_p0 = df_p0_raw.iloc[2:].copy()
df_p0.columns = ["region", "p0_persen"]
df_p0["p0_persen"] = pd.to_numeric(df_p0["p0_persen"], errors="coerce")
df_p0 = df_p0.dropna(subset=["p0_persen"])
# Hanya kab/kota (bukan provinsi — pola uppercase saja)
df_p0 = df_p0[~df_p0["region"].str.match(r'^[A-Z\s]+$', na=False)].copy()
df_p0["region"] = df_p0["region"].str.strip()
print(f"P0 2025: {len(df_p0)} kab/kota")

P0 2025: 515 kab/kota


In [3]:
# 2b. BPNT 2024 (Sumatera Selatan)
f_b24 = os.path.join(DATA_DIR, 
    "Percentage of Household Who Received Food Assistance (Non-Cash Food Assistance (BPNT)_Basic Food Program) by Regency_Municipality, 2024.xlsx")
df_b24 = pd.read_excel(f_b24)
df_b24 = df_b24.iloc[2:].copy()
df_b24.columns = ["region", "pct_bpnt"]
df_b24["pct_bpnt"] = pd.to_numeric(df_b24["pct_bpnt"], errors="coerce")
df_b24 = df_b24.dropna(subset=["pct_bpnt"])
df_b24["region"] = df_b24["region"].str.strip()
df_b24["tahun"] = 2024
print(f"BPNT 2024 (Sumsel): {len(df_b24)} wilayah")

BPNT 2024 (Sumsel): 18 wilayah


In [4]:
# 2c. BPNT + PKH 2023 (Jawa Tengah)
f_23 = os.path.join(DATA_DIR,
    "Percentage of Households who received Food Assistance (Non-Cash Food Assistance (BPNT)_Basic Food Program) and Program Keluarga Harapan (PKH) by Regency_Municipality, 2023.xlsx")
df_23 = pd.read_excel(f_23)
df_23 = df_23.iloc[3:].copy()
df_23.columns = ["region_raw", "pct_bpnt", "pct_pkh"]
df_23["pct_bpnt"] = pd.to_numeric(df_23["pct_bpnt"], errors="coerce")
df_23["pct_pkh"] = pd.to_numeric(df_23["pct_pkh"], errors="coerce")
df_23 = df_23.dropna(subset=["pct_bpnt", "pct_pkh"])
df_23["region"] = df_23["region_raw"].apply(lambda x: re.sub(r'^\d+\s+', '', str(x)).strip())
df_23 = df_23[df_23["region"].str.contains("Regency|Municipality", na=False)].copy()
df_23["region"] = df_23["region"].str.replace(" Regency", "", regex=False)
df_23["region"] = df_23["region"].str.replace(" Municipality", "", regex=False)
df_23["tahun"] = 2023
print(f"BPNT+PKH 2023 (Jateng): {len(df_23)} wilayah")

BPNT+PKH 2023 (Jateng): 35 wilayah


In [5]:
# 2d. BPNT + PKH 2021 (.xls — Jakarta)
f_21 = os.path.join(DATA_DIR,
    "percentage-of--household-who-received-food-assistance--non-cash-food-assistance--bpnt--basic-food-program--dan-program-keluarga-harapan--pkh--by-regency-city.xls")
df_21_raw = pd.read_excel(f_21)
rows = []
for i, row in df_21_raw.iterrows():
    val = str(row.iloc[0])
    m = re.match(r'^(\d+)\.\s*$', val.strip())
    if m:
        nama = str(row.iloc[2]).strip() if pd.notna(row.iloc[2]) else ""
        bpnt_val = row.iloc[4]
        pkh_val = row.iloc[6]
        if nama:
            rows.append({"region": nama, "pct_bpnt": pd.to_numeric(bpnt_val, errors="coerce"),
                         "pct_pkh": pd.to_numeric(pkh_val, errors="coerce")})
df_21 = pd.DataFrame(rows).dropna(subset=["pct_bpnt", "pct_pkh"])
df_21["tahun"] = 2021
print(f"BPNT+PKH 2021 (Jakarta): {len(df_21)} wilayah")

BPNT+PKH 2021 (Jakarta): 6 wilayah


---
## 3. Gabung Data & Merge dengan P0

In [6]:
real_bansos = pd.concat([df_b24, df_23, df_21], ignore_index=True)
real_bansos["region"] = real_bansos["region"].str.strip()

# Normalisasi nama untuk merge
def norm_name(n):
    n = n.lower().strip()
    n = re.sub(r'[^a-z\s]', '', n)
    n = re.sub(r'\s+', ' ', n)
    return n.strip()

df_p0["region_norm"] = df_p0["region"].apply(norm_name)
real_bansos["region_norm"] = real_bansos["region"].apply(norm_name)

merged = real_bansos.merge(
    df_p0[["region_norm", "p0_persen", "region"]],
    on="region_norm", how="left", suffixes=("", "_p0")
)

# Manual matching untuk record yang tidak match otomatis
no_match = merged[merged["p0_persen"].isna()]
if len(no_match) > 0:
    manual_map = {}
    for _, row in no_match.iterrows():
        r = row["region"]
        candidates = df_p0[df_p0["region"].str.contains(r[:10], case=False, na=False)]
        if len(candidates) > 0:
            manual_map[r] = candidates.iloc[0]["region"]
    for old, new in manual_map.items():
        mask = merged["region"].str.strip() == old
        p0_val = df_p0.loc[df_p0["region"] == new, "p0_persen"].values
        if len(p0_val) > 0 and merged.loc[mask, "p0_persen"].isna().any():
            merged.loc[mask, "p0_persen"] = p0_val[0]
            merged.loc[mask, "region"] = new

merged = merged.dropna(subset=["p0_persen"]).copy()

n_wilayah = merged["region"].nunique()
n_baris = len(merged)
print(f"Gabungan data real: {n_baris} baris, {n_wilayah} wilayah unik")
print(f"Tahun: {sorted(merged['tahun'].unique())}")
print(f"\nStatistik:")
print(merged[["p0_persen", "pct_bpnt", "pct_pkh"]].describe().round(2))

Gabungan data real: 56 baris, 51 wilayah unik
Tahun: [np.int64(2021), np.int64(2023), np.int64(2024)]

Statistik:
       p0_persen  pct_bpnt  pct_pkh
count      56.00     56.00    41.00
mean        9.34     19.58    17.42
std         2.87      9.62     6.92
min         3.23      2.67     2.07
25%         7.70     12.72    15.18
50%         9.52     17.59    18.53
75%        11.17     25.80    22.95
max        15.25     44.20    27.98


---
## 4. Feature Engineering

Karena data hanya memiliki 1 tahun per wilayah (tidak panel penuh 3 tahun untuk setiap wilayah), kita gunakan snapshot per-wilayah. Fitur:

1. **P0** — tingkat kemiskinan  
2. **pct_total_penerima** — BPNT (+ PKH jika tersedia)  
3. **rasio_ketepatan_sasaran** — total penerima / P0  
4. **diversity_index** — jumlah jenis bansos dengan cakupan >1%  
5. **composite_score** — skor gabungan (rasio * diversity) untuk memperkaya clustering

In [7]:
merged["pct_total_penerima"] = merged["pct_bpnt"].fillna(0) + merged["pct_pkh"].fillna(0)
merged["rasio_ketepatan_sasaran"] = merged["pct_total_penerima"] / merged["p0_persen"]
merged["diversity_index"] = (merged["pct_bpnt"].fillna(0) > 1).astype(int) + (merged["pct_pkh"].fillna(0) > 1).astype(int)

# Ambil snapshot: data paling terbaru per wilayah (prioritas tahun tertinggi)
snapshot = merged.sort_values("tahun", ascending=False).groupby("region").first().reset_index()

FEATURES = ["p0_persen", "pct_total_penerima", "rasio_ketepatan_sasaran", "diversity_index"]

df_feat = snapshot[["region"] + FEATURES].copy()
df_feat = df_feat.dropna()

print(f"Dataset final: {len(df_feat)} wilayah, {len(FEATURES)} fitur\n")
print(df_feat[FEATURES].describe().round(2))

# Tampilkan nama wilayah
print(f"\nWilayah: {df_feat['region'].tolist()}")

Dataset final: 51 wilayah, 4 fitur

       p0_persen  pct_total_penerima  rasio_ketepatan_sasaran  diversity_index
count      51.00               51.00                    51.00            51.00
mean        9.47               32.41                     3.51             1.73
std         2.93               16.83                     1.62             0.45
min         3.23                4.84                     0.44             1.00
25%         7.74               17.47                     2.04             1.00
50%         9.59               32.75                     3.77             2.00
75%        11.46               44.70                     4.40             2.00
max        15.25               71.00                     7.33             2.00

Wilayah: ['Banjarnegara', 'Banyumas', 'Batang', 'Blora', 'Boyolali', 'Brebes', 'Cilacap', 'Demak', 'Empat Lawang', 'Grobogan', 'Jepara', 'Karanganyar', 'Kebumen', 'Kendal', 'Kepulauan Seribu', 'Klaten', 'Kota Jakarta Barat', 'Kota Jakarta Pusat', 'Kota

---
## 5. Normalisasi

In [8]:
scaler = StandardScaler()
X = scaler.fit_transform(df_feat[FEATURES])
print(f"X shape: {X.shape}")

X shape: (51, 4)


---
## 6. Penentuan K Optimal (Elbow + Silhouette)

In [9]:
k_range = range(2, min(9, len(df_feat) // 2))
if len(k_range) < 2:
    k_range = range(2, 6)
k_range = list(k_range)

inertia, sil_scores = [], []
for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10).fit(X)
    inertia.append(km.inertia_)
    sil_scores.append(silhouette_score(X, km.labels_))

best_k = k_range[int(np.argmax(sil_scores))]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(k_range, inertia, marker="o", color=CLUSTER_COLORS[0], linewidth=2)
axes[0].set_title("Elbow Method")
axes[0].set_xlabel("Jumlah Cluster (K)")
axes[0].set_ylabel("Inertia")
axes[0].grid(alpha=0.3)

axes[1].plot(k_range, sil_scores, marker="o", color=CLUSTER_COLORS[1], linewidth=2)
axes[1].axvline(best_k, color="gray", linestyle="--", alpha=0.6)
axes[1].set_title(f"Silhouette Score (K optimal = {best_k})")
axes[1].set_xlabel("Jumlah Cluster (K)")
axes[1].set_ylabel("Silhouette Score")
axes[1].grid(alpha=0.3)

fig.suptitle("Penentuan Jumlah Cluster Optimal", y=1.03, fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/fig1_elbow_silhouette.png", bbox_inches="tight", dpi=200)
plt.close(fig)

print(f"Gambar: {OUTPUT_DIR}/fig1_elbow_silhouette.png")
print(f"K optimal = {best_k} (Silhouette = {max(sil_scores):.4f})")
for k, s in zip(k_range, sil_scores):
    print(f"  K={k}: Silhouette = {s:.4f}")

Gambar: /Users/budydjajani/Documents/outputs/outputs/fig1_elbow_silhouette.png
K optimal = 4 (Silhouette = 0.4604)
  K=2: Silhouette = 0.4517
  K=3: Silhouette = 0.4293
  K=4: Silhouette = 0.4604
  K=5: Silhouette = 0.4469
  K=6: Silhouette = 0.4062
  K=7: Silhouette = 0.4121
  K=8: Silhouette = 0.4276


---
## 7. Clustering & Perbandingan Algoritma

In [10]:
algorithms = {
    "K-Means": KMeans(n_clusters=best_k, random_state=RANDOM_SEED, n_init=10),
    "HAC (Ward)": AgglomerativeClustering(n_clusters=best_k, linkage="ward"),
    "DBSCAN": DBSCAN(eps=0.7, min_samples=3),
}

results = {}
metrics_rows = []
for name, algo in algorithms.items():
    labels = algo.fit_predict(X)
    n_clusters_found = len(set(labels)) - (1 if -1 in labels else 0)

    if n_clusters_found >= 2:
        sil = silhouette_score(X, labels)
        dbi = davies_bouldin_score(X, labels)
        ch = calinski_harabasz_score(X, labels)
    else:
        sil = dbi = ch = np.nan

    n_outlier = int((labels == -1).sum()) if name == "DBSCAN" else 0
    results[name] = labels
    metrics_rows.append(dict(
        algoritma=name, n_cluster=n_clusters_found, n_outlier=n_outlier,
        silhouette=round(sil, 4) if not np.isnan(sil) else "N/A",
        dbi=round(dbi, 4) if not np.isnan(dbi) else "N/A",
        calinski_harabasz=round(ch, 2) if not np.isnan(ch) else "N/A",
    ))

metrics_df = pd.DataFrame(metrics_rows)
print("Perbandingan Algoritma:")
print(metrics_df.to_string(index=False))
metrics_df.to_csv(f"{OUTPUT_DIR}/perbandingan_algoritma.csv", index=False)

Perbandingan Algoritma:
 algoritma  n_cluster  n_outlier  silhouette    dbi  calinski_harabasz
   K-Means          4          0      0.4604 0.7686              54.80
HAC (Ward)          4          0      0.4370 0.7814              50.99
    DBSCAN          4          7      0.3731 2.6414              18.99


In [11]:
# Visualisasi perbandingan metrik
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (col, title) in zip(axes, [
    ("silhouette", "Silhouette \u2191"),
    ("dbi", "Davies-Bouldin \u2193"),
    ("calinski_harabasz", "Calinski-Harabasz \u2191"),
]):
    vals = []
    for _, r in metrics_df.iterrows():
        try:
            vals.append(float(r[col]))
        except (ValueError, TypeError):
            vals.append(0)
    bars = ax.bar(metrics_df["algoritma"], vals, color=CLUSTER_COLORS[:len(metrics_df)])
    ax.set_title(title, fontsize=10)
    ax.tick_params(axis="x", rotation=15)
    ax.grid(axis="y", alpha=0.3)
    for bar, v in zip(bars, vals):
        if v != 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                    f"{v:.3f}", ha="center", va="bottom", fontsize=7)

fig.suptitle("Perbandingan Algoritma Clustering", y=1.04, fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/fig2_perbandingan_algoritma.png", bbox_inches="tight", dpi=200)
plt.close(fig)
print(f"Gambar: {OUTPUT_DIR}/fig2_perbandingan_algoritma.png")

Gambar: /Users/budydjajani/Documents/outputs/outputs/fig2_perbandingan_algoritma.png


---
## 8. Visualisasi PCA 2D

In [12]:
pca = PCA(n_components=min(2, X.shape[1]), random_state=RANDOM_SEED)
coords = pca.fit_transform(X)
var_explained = pca.explained_variance_ratio_ * 100

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (name, labels) in zip(axes, results.items()):
    unique_labels = sorted(set(labels))
    for i, lab in enumerate(unique_labels):
        mask = labels == lab
        color = "#9CA3AF" if lab == -1 else CLUSTER_COLORS[i % len(CLUSTER_COLORS)]
        marker = "x" if lab == -1 else "o"
        label_text = "Anomali" if lab == -1 else f"Cluster {lab}"
        ax.scatter(coords[mask, 0], coords[mask, 1], s=40, alpha=0.7,
                   color=color, marker=marker, label=label_text, edgecolors="none")
    ax.set_title(name, fontsize=11)
    ax.set_xlabel(f"PC1 ({var_explained[0]:.1f}%)")
    ax.set_ylabel(f"PC2 ({var_explained[1]:.1f}%)")
    ax.legend(fontsize=7, loc="best", framealpha=0.9)
    ax.grid(alpha=0.25)

fig.suptitle("Visualisasi Cluster pada Ruang PCA 2D", y=1.03, fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/fig3_pca_clusters.png", bbox_inches="tight", dpi=200)
plt.close(fig)
print(f"Gambar: {OUTPUT_DIR}/fig3_pca_clusters.png")
print(f"Varians: PC1={var_explained[0]:.1f}%, PC2={var_explained[1]:.1f}%")

Gambar: /Users/budydjajani/Documents/outputs/outputs/fig3_pca_clusters.png


Varians: PC1=58.7%, PC2=32.2%


---
## 9. Profil & Interpretasi Cluster (K-Means)

In [13]:
kmeans_labels = results["K-Means"]
df_result = df_feat.copy()
df_result["cluster"] = kmeans_labels

profile = df_result.groupby("cluster")[FEATURES].mean().round(2)
profile["jumlah_wilayah"] = df_result.groupby("cluster").size()
profile.to_csv(f"{OUTPUT_DIR}/profil_cluster.csv")
print(f"CSV: {OUTPUT_DIR}/profil_cluster.csv")

CSV: /Users/budydjajani/Documents/outputs/outputs/profil_cluster.csv


In [14]:
# Visualisasi profil (z-score)
scaler_z = StandardScaler()
z = pd.DataFrame(scaler_z.fit_transform(df_result[FEATURES]), columns=FEATURES)
z["cluster"] = df_result["cluster"].values
z_profile = z.groupby("cluster").mean()

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(FEATURES))
width = 0.8 / max(len(z_profile), 1)
for i, (cl, row) in enumerate(z_profile.iterrows()):
    offset = (i - (len(z_profile) - 1) / 2) * width
    ax.bar(x + offset, row.values, width=width, label=f"Cluster {cl}",
           color=CLUSTER_COLORS[i % len(CLUSTER_COLORS)])

labels_ft = ["P0\n(Kemiskinan)", "%Total\nPenerima", "Rasio\nKetepatan", "Diversity\nIndex"]
ax.set_xticks(x)
ax.set_xticklabels(labels_ft, fontsize=9)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("Nilai Standardisasi (Z-score)")
ax.set_title("Profil Karakteristik Tiap Cluster (K-Means)", fontsize=13, fontweight="bold")
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/fig4_profil_cluster.png", bbox_inches="tight", dpi=200)
plt.close(fig)
print(f"Gambar: {OUTPUT_DIR}/fig4_profil_cluster.png")

Gambar: /Users/budydjajani/Documents/outputs/outputs/fig4_profil_cluster.png


In [15]:
# Label deskriptif per cluster
labels_map = {}
for cl, row in profile.iterrows():
    rasio = row["rasio_ketepatan_sasaran"]
    p0 = row["p0_persen"]
    n = int(row["jumlah_wilayah"])
    if rasio < 0.75:
        desc = "Under-coverage" if p0 >= 7 else "Tepat Sasaran - Kemiskinan Rendah"
    elif rasio > 2.0:
        desc = "Over-coverage (indikasi kebocoran/salah sasaran)"
    elif rasio > 1.5:
        desc = "Tepat Sasaran - Kemiskinan Sedang/Tinggi" if p0 >= 7 else "Tepat Sasaran - Kemiskinan Rendah"
    else:
        desc = "Tepat Sasaran - Kemiskinan Sedang/Tinggi" if p0 >= 7 else "Tepat Sasaran - Kemiskinan Rendah"
    labels_map[cl] = (desc, n)
    print(f"Cluster {cl} ({n} wilayah): {desc}")
    print(f"  - P0 rata-rata: {p0:.2f}%")
    print(f"  - Rasio ketepatan: {rasio:.2f}\n")

Cluster 0 (5 wilayah): Over-coverage (indikasi kebocoran/salah sasaran)
  - P0 rata-rata: 4.42%
  - Rasio ketepatan: 2.15

Cluster 1 (18 wilayah): Over-coverage (indikasi kebocoran/salah sasaran)
  - P0 rata-rata: 11.67%
  - Rasio ketepatan: 4.15

Cluster 2 (14 wilayah): Tepat Sasaran - Kemiskinan Sedang/Tinggi
  - P0 rata-rata: 10.69%
  - Rasio ketepatan: 1.73

Cluster 3 (14 wilayah): Over-coverage (indikasi kebocoran/salah sasaran)
  - P0 rata-rata: 7.22%
  - Rasio ketepatan: 4.95



---
## 10. Scatter Plot: Rasio Ketepatan vs P0

In [16]:
fig, ax = plt.subplots(figsize=(8, 6))
unique_labels = sorted(set(kmeans_labels))
for i, lab in enumerate(unique_labels):
    mask = kmeans_labels == lab
    color = "#9CA3AF" if lab == -1 else CLUSTER_COLORS[i % len(CLUSTER_COLORS)]
    marker = "x" if lab == -1 else "o"
    label_text = "Anomali" if lab == -1 else f"Cluster {lab}"
    ax.scatter(df_result.loc[mask, "p0_persen"], df_result.loc[mask, "rasio_ketepatan_sasaran"],
               s=40, alpha=0.7, color=color, marker=marker, label=label_text)

ax.axhline(1.0, color="red", linestyle="--", alpha=0.5, linewidth=1)
ax.axhline(1.5, color="orange", linestyle=":", alpha=0.5, linewidth=1)
ax.axhline(0.75, color="orange", linestyle=":", alpha=0.5, linewidth=1)
ax.text(0.02, 0.95, "Over-coverage", transform=ax.transAxes, fontsize=9, color="orange", va="top")
ax.text(0.02, 0.70, "Proporsional", transform=ax.transAxes, fontsize=9, color="green", va="top")
ax.text(0.02, 0.10, "Under-coverage", transform=ax.transAxes, fontsize=9, color="orange", va="top")
ax.set_xlabel("Persentase Penduduk Miskin (P0) %")
ax.set_ylabel("Rasio Ketepatan Sasaran (%penerima / %kemiskinan)")
ax.set_title("Rasio Ketepatan Sasaran vs Tingkat Kemiskinan (K-Means)", fontsize=12, fontweight="bold")
ax.legend(fontsize=8)
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/fig5_rasio_vs_p0.png", bbox_inches="tight", dpi=200)
plt.close(fig)
print(f"Gambar: {OUTPUT_DIR}/fig5_rasio_vs_p0.png")

Gambar: /Users/budydjajani/Documents/outputs/outputs/fig5_rasio_vs_p0.png


---
## 11. Deteksi Anomali DBSCAN & Simpan Hasil

In [17]:
db_labels = results["DBSCAN"]
n_anomali = int((db_labels == -1).sum())
print(f"Anomali DBSCAN: {n_anomali} dari {len(db_labels)} ({n_anomali/len(db_labels)*100:.1f}%)")

df_result["cluster_label"] = df_result["cluster"].map({k: v[0] for k, v in labels_map.items()})
df_result["dbscan_anomali"] = (db_labels == -1)
df_result.to_csv(f"{OUTPUT_DIR}/hasil_clustering_wilayah.csv", index=False)

print(f"\nSemua output di: '{OUTPUT_DIR}/'")
print("=" * 60)
print("  SELESAI!")
print(f"  - 5 gambar PNG (fig1-fig5)")
print(f"  - 3 file CSV")
print("=" * 60)

Anomali DBSCAN: 7 dari 51 (13.7%)

Semua output di: '/Users/budydjajani/Documents/outputs/outputs/'
  SELESAI!
  - 5 gambar PNG (fig1-fig5)
  - 3 file CSV


---
## 12. Ringkasan Hasil

**Data:** 51 kabupaten/kota dari Sumatera Selatan (2024), Jawa Tengah (2023), dan DKI Jakarta (2021).

**K optimal:** 4 (Silhouette = 0.4604)

| Algoritma | Silhouette | DBI | Calinski-Harabasz |
|---|---|---|---|
| K-Means (K=4) | 0.4604 | 0.7686 | 54.80 |
| HAC (Ward) (K=4) | 0.4370 | 0.7814 | 50.99 |
| DBSCAN (eps=0.7) | 0.3731 | 2.6414 | 18.99 |

**Interpretasi Cluster (K-Means):**
- **Cluster 0** (5 wilayah): Over-coverage — P0 rendah (4.42%), rasio 2.15
- **Cluster 1** (18 wilayah): Over-coverage tinggi — P0 tinggi (11.67%), rasio 4.15
- **Cluster 2** (14 wilayah): Tepat Sasaran — P0 tinggi (10.69%), rasio 1.73
- **Cluster 3** (14 wilayah): Over-coverage — P0 sedang (7.22%), rasio 4.95

**Temuan:** Seluruh wilayah dalam sampel menunjukkan rasio ketepatan >1.5 yang mengindikasikan cakupan bansos melebihi proporsi penduduk miskin — pola konsisten di Sumsel, Jateng, dan Jakarta.

**DBSCAN:** 7 wilayah (13.7%) teridentifikasi sebagai anomali — prioritas verifikasi lapangan.